In [1]:
# Import libraries
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

C:\Users\aksha\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\aksha\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [7]:
df = pd.read_csv("C:/Users/aksha/Documents/Github Datasets/Airbnb/Dataset.csv")
df = df.iloc[:, :5]
df = df.dropna(subset=['Reviews per Month'])
df

,id,Property Type,Instant Bookable,Reviews per Month,Listing Location
0,2.708000e+03,Private room in rental unit,t,0.33,Los Angeles
1,2.732000e+03,Private room in rental unit,f,0.14,Los Angeles
3,6.033000e+03,Entire bungalow,f,0.10,Los Angeles
4,6.931000e+03,Private room in rental unit,t,0.19,Los Angeles
5,7.874000e+03,Private room in home,f,0.24,Los Angeles
...,...,...,...,...,...
220031,1.488560e+18,Entire home,t,3.00,Asheville
220035,1.492190e+18,Entire home,f,2.00,Asheville
220036,1.492220e+18,Entire rental unit,t,1.00,Asheville
220041,1.502160e+18,Private room in home,f,1.00,Asheville


In [14]:
import pandas as pd
from scipy.stats import ttest_ind

# Clean column names (important if spaces exist)
df.columns = df.columns.str.strip().str.replace(' ', '_')

# Convert Instant_Bookable to binary (optional but helpful)
df['Instant_Bookable_flag'] = (df['Instant_Bookable'] == 't').astype(int)

# Drop NA in Reviews_per_Month
df = df.dropna(subset=['Reviews_per_Month'])

# -----------------------------
# 1. T-test for ALL cities
# -----------------------------
group_t = df[df['Instant_Bookable'] == 't']['Reviews_per_Month']
group_f = df[df['Instant_Bookable'] == 'f']['Reviews_per_Month']

t_stat, p_value = ttest_ind(group_t, group_f, equal_var=False)

print("Overall T-test:")
print("T-stat:", t_stat)
print(f"P-value (all cities): {p_value:.16f}")


# -----------------------------
# 2. T-test BY CITY
# -----------------------------
results = []

for city, data in df.groupby('Listing_Location'):
    
    group_t = data[data['Instant_Bookable'] == 't']['Reviews_per_Month']
    group_f = data[data['Instant_Bookable'] == 'f']['Reviews_per_Month']
    
    # Only run test if both groups exist
    if len(group_t) > 1 and len(group_f) > 1:
        t_stat, p_value = ttest_ind(group_t, group_f, equal_var=False)
        
        results.append({
            'city': city,
            't_stat': t_stat,
            'p_value': p_value,
            'mean_t': group_t.mean(),
            'mean_f': group_f.mean()
        })

results_df = pd.DataFrame(results)

results_df['p_value'] = results_df['p_value'].apply(lambda x: f"{x:.16f}")

print("\nT-test by city:")
print(results_df)

Overall T-test:
T-stat: 35.84133174821933
P-value (all cities): 0.0000000000000000

T-test by city:
             city     t_stat             p_value    mean_t    mean_f
0       Asheville   2.485955  0.0130079846577899  2.306117  2.099102
1          Austin  15.220591  0.0000000000000000  2.101052  1.494335
2          Boston   4.693352  0.0000028398455287  1.885224  1.553677
3         Chicago   2.162557  0.0306487079525348  2.045708  1.906044
4        Columbus   0.956914  0.3387303136624318  2.192818  2.116584
5          Dallas   6.351330  0.0000000002382888  2.228068  1.861771
6          Denver   7.727704  0.0000000000000181  2.308945  1.693687
7      Fort Worth   4.356002  0.0000144638821777  2.084966  1.672941
8          Hawaii  -8.961553  0.0000000000000000  1.071111  1.212666
9     Los Angeles  20.886033  0.0000000000000000  1.762903  1.228302
10      Nashville   3.738337  0.0001865205799396  2.302781  2.107916
11    New Orleans   6.428050  0.0000000001411582  1.704055  1.421271
12 